# P. putida KT2440: fitness vs expression (Figure S5)

Analysis behind Supplementary Figure S5.

**Expression data:** GEO [GSE85475](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE85475), control samples `C_1_0`, `C_2_0`, `C_3_0`
(KT2440 in minimal medium with glucose).
TPM computed by `scripts/tpm_from_counts.py` from raw counts, CDS length, `PP_` genes only (tRNA/rRNA excluded).
Output: `data/processed/GSE85475_TPM.tsv`.

In [1]:
import numpy as np
import pandas as pd

## Step 1. Load TPM (control samples)

In [2]:
def load_tpm(file, samples):
    """TPM table -> locus_tag, TPM per replicate, mean TPM and log10(TPM + 1)."""
    cols = [f'TPM_{s}' for s in samples]
    df = pd.read_csv(file, sep='\t', usecols=['locus_tag'] + cols)
    df = df.assign(TPM=df[cols].mean(axis=1))
    df = df.assign(TPM_log10=np.log10(df.TPM + 1))
    return df

In [3]:
file    = './data/processed/GSE85475_TPM.tsv'
samples = ['C_1_0', 'C_2_0', 'C_3_0']
TPMs    = load_tpm(file, samples)
TPMs.head(2)

,locus_tag,TPM_C_1_0,TPM_C_2_0,TPM_C_3_0,TPM,TPM_log10
0,PP_0001,180.860403,167.241775,164.560274,170.887484,2.235244
1,PP_0002,230.368655,197.890775,215.707386,214.655605,2.333761


In [4]:
print(f'Genes: {len(TPMs)}')
print('TPM sum per replicate:', TPMs[[f'TPM_{s}' for s in samples]].sum().round(1).to_dict())
TPMs.TPM_log10.describe()

Genes: 5350
TPM sum per replicate: {'TPM_C_1_0': 1000000.0, 'TPM_C_2_0': 1000000.0, 'TPM_C_3_0': 1000000.0}


count    5350.000000
mean        1.400648
std         0.748657
min         0.000000
25%         0.824848
50%         1.372736
75%         1.875798
max         4.936240
Name: TPM_log10, dtype: float64

## Step 2. Merge with fitness

**Fitness data:** `data/raw/fit_organism_Putida.tsv` (Fitness Browser), column `set6IT057 D-Glucose (C)` only.
Note: set6IT057 correlates 0.61–0.70 with the other glucose experiments in the file (set1, set5, set12; 0.76–0.92 among themselves).

Categories: E essential (≤ −2) · I important (−2, −0.5) · N mean-effect [−0.5, 0.5] · F fitness-enhancing (> 0.5).

In [5]:
def load_fitness(file, columns):
    """Fitness Browser table -> locusId and D_Glucose (mean of the given columns)."""
    df = pd.read_csv(file, sep='\t', usecols=['locusId'] + columns)
    df = df.assign(D_Glucose=df[columns].mean(axis=1))
    return df[['locusId', 'D_Glucose']]


def category_fitness(df, col='D_Glucose'):
    """Add <col>_cat: E (<= -2), I (-2, -0.5), N [-0.5, 0.5], F (> 0.5)."""
    x = df[col]
    cat = np.select([x <= -2, x < -0.5, x <= 0.5, x > 0.5], ['E', 'I', 'N', 'F'], default='x')
    return df.assign(**{f'{col}_cat': cat})


def merge_fitness_tpm(fit, tpm):
    """Inner join on locusId = locus_tag; report genes lost on each side."""
    only_fit = sorted(set(fit.locusId) - set(tpm.locus_tag))
    only_tpm = sorted(set(tpm.locus_tag) - set(fit.locusId))
    df = fit.merge(tpm, how='inner', left_on='locusId', right_on='locus_tag')
    print(f'Fitness: {len(fit)}  TPM: {len(tpm)}  Merged: {len(df)}')
    print(f'Only in fitness: {len(only_fit)}  e.g. {only_fit[:10]}')
    print(f'Only in TPM:     {len(only_tpm)}  e.g. {only_tpm[:10]}')
    return df

In [6]:
file        = './data/raw/fit_organism_Putida.tsv'
fit_columns = ['set6IT057 D-Glucose (C)']
dtaFit      = category_fitness(load_fitness(file, fit_columns))
dtaFit.D_Glucose_cat.value_counts()

N    4503
I     158
F      71
E      46
Name: D_Glucose_cat, dtype: int64

In [7]:
fitness_tpm = merge_fitness_tpm(dtaFit, TPMs)
fitness_tpm.head(2)

Fitness: 4778  TPM: 5350  Merged: 4477
Only in fitness: 301  e.g. ['PP_0156', 'PP_0489', 'PP_0511', 'PP_0598', 'PP_1409', 'PP_1731', 'PP_1906', 'PP_1994', 'PP_2110', 'PP_3333']
Only in TPM:     873  e.g. ['PP_0004', 'PP_0005', 'PP_0006', 'PP_0008', 'PP_0009', 'PP_0010', 'PP_0011', 'PP_0013', 'PP_0018', 'PP_0019']


,locusId,D_Glucose,D_Glucose_cat,locus_tag,TPM_C_1_0,TPM_C_2_0,TPM_C_3_0,TPM,TPM_log10
0,PP_0001,-0.276,N,PP_0001,180.860403,167.241775,164.560274,170.887484,2.235244
1,PP_0002,-0.124,N,PP_0002,230.368655,197.890775,215.707386,214.655605,2.333761


## Step 3. Expression cutoff

Highly expressed = top 5% of log10(TPM + 1) among the merged genes (fitness ∩ TPM),
the same threshold used for E. coli. A percentile does not depend on the TPM scale. Selected genes: mean-effect category (N) and above the cutoff.

In [8]:
def percentile_cutoff(values, top_fraction):
    """Values -> threshold that keeps the top fraction."""
    return values.quantile(1 - top_fraction)


def select_genes(df, cutoff, category='N', col='TPM_log10', cat_col='D_Glucose_cat'):
    """Genes in the fitness category with expression >= cutoff."""
    return df[(df[cat_col] == category) & (df[col] >= cutoff)]

In [9]:
TOP_FRACTION = 0.05
cutoff   = percentile_cutoff(fitness_tpm.TPM_log10, TOP_FRACTION)
above    = fitness_tpm[fitness_tpm.TPM_log10 >= cutoff]
selected = select_genes(fitness_tpm, cutoff)

print(f'Cutoff (top {TOP_FRACTION:.1%}): log10(TPM + 1) >= {cutoff:.3f}  (TPM >= {10**cutoff - 1:,.0f})')
print(f'Genes above cutoff: {len(above)}  ->  by category:', above.D_Glucose_cat.value_counts().to_dict())
print(f'Selected (N, above cutoff): {len(selected)}')

Cutoff (top 5.0%): log10(TPM + 1) >= 2.400  (TPM >= 250)
Genes above cutoff: 224  ->  by category: {'N': 168, 'I': 27, 'E': 16, 'F': 13}
Selected (N, above cutoff): 168


## Step 4. Panel A: fitness vs expression

Scatter of fitness (x) vs log10(TPM + 1) (y), colored by fitness category, with marginal histograms.
Dotted lines: fitness −2, −0.5, 0.5 and the expression cutoff from Step 3.
Export: camera button in the plot toolbar (PNG).

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

config = {
    'toImageButtonOptions': {'format': 'png'},
    'scrollZoom': False,
    'displayModeBar': True,
    'staticPlot': False,
    'displaylogo': False,
    'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'zoomIn', 'zoomOut',
                               'autoScale', 'resetScale', 'lasso2d'],
}

COLOR_MAP = {'E': 'red', 'I': 'orange', 'N': 'blue', 'F': 'green'}


def plot_fitness_expression(df, cutoff, fsize=12, width=1000, height=570):
    """Fitness/TPM table + expression cutoff -> scatter with marginal histograms."""
    fig = make_subplots(
        rows=2, cols=2,
        shared_xaxes=True, shared_yaxes=True,
        vertical_spacing=0.00, horizontal_spacing=0.00,
        column_widths=[0.8, 0.2], row_heights=[0.2, 0.8],
        specs=[[{"type": "histogram"}, {}],
               [{"type": "scatter"}, {"type": "histogram"}]]
    )

    # Scatter
    fig.add_trace(go.Scatter(
        x=df.D_Glucose, y=df.TPM_log10, mode='markers',
        marker=dict(color=[COLOR_MAP[c] for c in df.D_Glucose_cat], size=5, opacity=0.9,
                    line=dict(width=0.5, color='DarkSlateGrey')),
        customdata=df.locusId,
        hovertemplate='<b>LocusId:</b> %{customdata}<br><b>Fitness:</b> %{x:.2f}<br>'
                      '<b>TPM_log10:</b> %{y:.2f}<extra></extra>',
    ), row=2, col=1)

    # Marginal histograms
    fig.add_trace(go.Histogram(x=df.D_Glucose, nbinsx=800, marker_color='dodgerblue',
                               showlegend=False), row=1, col=1)
    fig.add_trace(go.Histogram(y=df.TPM_log10, nbinsy=100, marker_color='dodgerblue',
                               showlegend=False), row=2, col=2)

    # Reference lines
    fig.add_hline(y=0, line_dash='solid', line_color='black', line_width=0.5, row=2, col=1)
    fig.add_vline(x=-7, line_dash='solid', line_color='black', line_width=0.5, row=2, col=1)
    fig.add_hline(y=cutoff, line_dash='dot', line_color='gray', line_width=0.9, row=2, col=1)
    for x in (0.5, -0.5, -2):
        fig.add_vline(x=x, line_dash='dot', line_color='gray', line_width=0.9, row=2, col=1)

    # Category labels, above the highest point
    y_max = np.ceil(df.TPM_log10.max() * 2) / 2 + 0.5
    label_y = y_max - 0.15
    for text, x, anchor in (('essential', -4.5, 'center'), ('important', -1.25, 'center'),
                            ('mean-effect', 0, 'center'), ('fitness-enhancing', 0.55, 'left')):
        fig.add_annotation(text=text, x=x, y=label_y, xanchor=anchor, showarrow=False,
                           font=dict(color='black', size=fsize), row=2, col=1)

    fig.update_layout(width=width, height=height, plot_bgcolor='rgba(0, 0, 0, 0)',
                      showlegend=False, font=dict(color='black', size=14),
                      margin=dict(t=20, b=60, l=80, r=20))
    fig.update_xaxes(title_text='Fitness', row=2, col=1)
    fig.update_yaxes(title_text='log<sub>10</sub>(TPM + 1)', range=[-0.2, y_max], row=2, col=1)
    for r, c in ((1, 1), (2, 2)):
        fig.update_xaxes(showticklabels=False, row=r, col=c)
        fig.update_yaxes(showticklabels=False, row=r, col=c)
    return fig

In [11]:
figA = plot_fitness_expression(fitness_tpm, cutoff)
figA.show(config=config)

## Step 5. Panel B: COG categories of the selected genes

Annotation: NCBI feature table (`GCA_000007565`) + COGs (`data/raw/COGs.csv`).
Only genes with a single COG letter are counted; genes without COG
or with several letters (e.g. `K,H`) are reported but not plotted.

In [12]:
def load_genome(file):
    """NCBI feature table -> one row per gene with coordinates, product name and symbol."""
    df = pd.read_table(file)
    gene = df.loc[df['# feature'] == 'gene', ['locus_tag', 'start', 'end', 'strand']]
    cds = df.loc[df['# feature'] == 'CDS', ['locus_tag', 'name', 'symbol', 'product_length']]
    return gene.merge(cds, how='left', on='locus_tag')


def add_COGs(file, genome):
    """Add cogs and func (COG letters) from the COG table."""
    cogs = pd.read_table(file).rename(columns={'bnum': 'locus_tag'})
    return genome.merge(cogs, how='left', on='locus_tag')


def count_cogs(df, col='func'):
    """Annotated genes -> (genes per single COG letter, n without COG, list of multi-COG values)."""
    no_cog = df[col].isna()
    multi = df[col].str.contains(',', na=False)
    print(f'Genes: {len(df)}  counted: {(~no_cog & ~multi).sum()}  '
          f'without COG: {no_cog.sum()}  multiple COGs: {multi.sum()} {sorted(df.loc[multi, col])}')
    counts = df.loc[~no_cog & ~multi, col].value_counts().rename_axis('cog').reset_index(name='n_genes')
    return counts, int(no_cog.sum()), sorted(df.loc[multi, col])

In [13]:
COG_TABLE = pd.DataFrame({
    'COG':   list('JAKLBDYVTMNZWUOXCGEFHIPQRS'),
    'group': ['INFORMATION'] * 5 + ['CELLULAR PROCESSES'] * 11 + ['METABOLISM'] * 8 +
             ['POORLY CHARACTERIZED'] * 2,
    'name':  ['Translation', 'RNA processing', 'Transcription', 'Replication', 'Chromatin structure',
              'Cell division', 'Nuclear structure', 'Defense mechanisms', 'Signal transduction',
              'Cell wall', 'Cell motility', 'Cytoskeleton', 'Extracellular structures',
              'Intracellular trafficking', 'Posttranslational', 'Mobilome', 'Energy',
              'Carbohydrate', 'Amino acid', 'Nucleotide', 'Coenzyme', 'Lipid', 'Inorganic ion',
              'Secondary metabolites', 'General function', 'Function unknown'],
})
GROUP_ORDER = ['INFORMATION', 'CELLULAR PROCESSES', 'METABOLISM', 'POORLY CHARACTERIZED']
GROUP_COLOR = {'INFORMATION': '#B22222', 'CELLULAR PROCESSES': '#1E90FF',
               'METABOLISM': '#19C918', 'POORLY CHARACTERIZED': '#D3D3D3'}


def plot_cog_bars(counts, width=900, height=480):
    """COG counts (cog, n_genes) -> horizontal bars grouped by functional group, with totals."""
    df = COG_TABLE.merge(counts, left_on='COG', right_on='cog', how='inner')
    df['group'] = pd.Categorical(df['group'], categories=GROUP_ORDER, ordered=True)
    df = df.sort_values('group', kind='stable')
    groups = [g for g in GROUP_ORDER if g in set(df['group'])]

    fig = go.Figure()
    for g in groups:
        d = df[df['group'] == g]
        fig.add_trace(go.Bar(y=d['name'], x=d['n_genes'], orientation='h', name=g,
                             marker=dict(color=GROUP_COLOR[g], opacity=0.85,
                                         line=dict(width=1, color='rgba(0,0,0,0.2)')),
                             text=d['n_genes'], textposition='outside',
                             textfont=dict(size=14, color='black', family='Arial'),
                             hovertemplate='<b>%{y}</b><br>Genes: %{x}<extra></extra>'))

    # Group separators, labels and totals on the right margin
    pos, annotations = 0, []
    for g in groups:
        n = (df['group'] == g).sum()
        if pos > 0:
            fig.add_hline(y=pos - 0.5, line_dash='dot', line_color='rgba(0,0,0,0.3)', line_width=1)
        mid = pos + n / 2 - 0.5
        annotations += [
            dict(xref='paper', yref='y', x=1.02, y=mid, text=f'<b>{g}</b>', showarrow=False,
                 font=dict(size=14, color=GROUP_COLOR[g], family='Arial'), xanchor='left'),
            dict(xref='paper', yref='y', x=1.02, y=mid - 0.8,
                 text=f"Total: <b>{int(df.loc[df['group'] == g, 'n_genes'].sum())}</b>",
                 showarrow=False, font=dict(size=14, color=GROUP_COLOR[g], family='Arial'),
                 xanchor='left'),
        ]
        pos += n

    fig.update_layout(
        yaxis=dict(title='<b>COG categories</b>', autorange='reversed', tickfont=dict(size=14)),
        xaxis=dict(title='<b>Number of genes</b>', range=[0, df['n_genes'].max() * 1.35],
                   gridcolor='rgba(230,230,230,0.7)', tickfont=dict(size=14)),
        showlegend=False, height=height, width=width,
        margin=dict(l=150, r=230, b=60, t=20, pad=10),
        paper_bgcolor='white', plot_bgcolor='white', annotations=annotations,
    )
    return fig

In [14]:
genome = load_genome('./data/raw/GCA_000007565_feature_table.txt')
genome = add_COGs('./data/raw/COGs.csv', genome)

selected_info = selected.merge(genome, how='left', left_on='locusId', right_on='locus_tag')
cog_counts, n_no_cog, multi_cogs = count_cogs(selected_info)
cog_counts

Genes: 168  counted: 123  without COG: 43  multiple COGs: 2 ['K,H', 'M,N']


,cog,n_genes
0,M,16
1,E,13
2,K,13
3,O,12
4,J,11
5,G,9
6,S,7
7,T,6
8,P,6
9,Q,5


In [15]:
figB = plot_cog_bars(cog_counts)
figB.show(config=config)

In [16]:
def build_table(selected_info):
    """Selected genes + annotation -> Table C (locusId, Symbol, Name, COG, Fitness, log10(TPM+1))."""
    table = selected_info[['locusId', 'symbol', 'name', 'func', 'D_Glucose', 'TPM_log10']]
    table.columns = ['locusId', 'Symbol', 'Name', 'COG', 'Fitness', 'log10(TPM+1)']
    return table.sort_values('locusId').reset_index(drop=True)

In [17]:
tableC = build_table(selected_info)
tableC.to_csv('./results/FigureS5_tableC.tsv', sep='\t', index=False)
print(f'Table C: {len(tableC)} genes -> results/FigureS5_tableC.tsv')
tableC

Table C: 168 genes -> results/FigureS5_tableC.tsv


,locusId,Symbol,Name,COG,Fitness,log10(TPM+1)
0,PP_0086,NaN,conserved protein of unknown function,R,0.227,2.636964
1,PP_0168,NaN,putative surface adhesion protein,Q,0.307,2.532776
2,PP_0181,NaN,conserved protein of unknown function,NaN,0.039,2.690576
3,PP_0191,algQ,Transcriptional regulatory protein AlgQ,K,0.281,2.508312
4,PP_0194,algP,Transcriptional regulatory protein AlgP,NaN,-0.234,2.414377
...,...,...,...,...,...,...
163,PP_5193,gcvH-II,glycine cleavage system H protein 2,E,-0.249,2.482177
164,PP_5204,NaN,conserved protein of unknown function,NaN,-0.238,2.811562
165,PP_5313,hupA,DNA-binding protein HU-alpha,L,0.260,2.672933
166,PP_5353,NaN,conserved exported protein of unknown function,I,0.173,2.493735


## Step 7. Supplementary PDF (Letter, 2 pages)

Page 1: panels A and B (PNG exported from plotly at 300 dpi-equivalent).
Page 2: Table C as real text (searchable), two columns.
Output: `results/FigureS5.pdf`. Requires Chrome for kaleido (`plotly_get_chrome`).

In [18]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader
from reportlab.pdfbase.pdfmetrics import stringWidth
from reportlab.pdfgen import canvas
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import Paragraph


def export_png(fig, file, scale=3):
    """Plotly figure -> PNG file; returns the file path."""
    fig.write_image(file, scale=scale, width=fig.layout.width, height=fig.layout.height)  # kaleido 1.x ignores layout size
    return file


DESC_STYLE = ParagraphStyle('desc', fontName='Helvetica', fontSize=8, leading=10)


def draw_panels(c, images, descriptions, margin=36, gap=14):
    """Stack images vertically with a description under each; scale to fit the page."""
    page_w, page_h = letter
    readers = [ImageReader(i) for i in images]
    ratios = [r.getSize()[1] / r.getSize()[0] for r in readers]
    paras = [Paragraph(d, DESC_STYLE) for d in descriptions]
    text_w = page_w - 2 * margin
    text_h = [p.wrap(text_w, page_h)[1] for p in paras]
    free_h = page_h - 2 * margin - sum(text_h) - gap * (2 * len(images) - 1)
    width = min(text_w, free_h / sum(ratios))
    y = page_h - margin
    for r, ratio, para, th in zip(readers, ratios, paras, text_h):
        h = width * ratio
        c.drawImage(r, (page_w - width) / 2, y - h, width, h)
        y -= h + gap / 2
        para.drawOn(c, margin, y - th)
        y -= th + gap


def fit_text(text, width, font, size):
    """Truncate text with an ellipsis so it fits in width."""
    if stringWidth(text, font, size) <= width:
        return text, False
    while text and stringWidth(text + '…', font, size) > width:
        text = text[:-1]
    return text + '…', True


def draw_table(c, table, title, margin=30, gap=12, size=5.6):
    """Draw the table in two side-by-side columns on one page; returns n truncated names."""
    page_w, page_h = letter
    font, bold = 'Helvetica', 'Helvetica-Bold'
    headers = list(table.columns)
    col_w = [30, 26, 0, 18, 26, 38]                       # Name width computed below
    half_w = (page_w - 2 * margin - gap) / 2
    col_w[2] = half_w - sum(col_w)
    right = {4, 5}                                        # numeric columns, right-aligned

    c.setFont(bold, 9)
    c.drawString(margin, page_h - margin, title)
    top = page_h - margin - 14
    n_rows = int(np.ceil(len(table) / 2))
    row_h = (top - margin) / (n_rows + 1)

    rows = table.fillna('').astype(str).values.tolist()
    truncated = 0
    for half in range(2):
        x0 = margin + half * (half_w + gap)
        chunk = rows[half * n_rows:(half + 1) * n_rows]
        for i, row in enumerate([headers] + chunk):
            y = top - (i + 1) * row_h + row_h * 0.25
            c.setFont(bold if i == 0 else font, size)
            x = x0
            for j, (val, w) in enumerate(zip(row, col_w)):
                if j == 2 and i > 0:
                    val, cut = fit_text(val, w - 2, font, size)
                    truncated += cut
                if j in right:
                    c.drawRightString(x + w - 2, y, val)
                else:
                    c.drawString(x, y, val)
                x += w
            if i == 0:
                c.setLineWidth(0.4)
                c.line(x0, y - row_h * 0.3, x0 + half_w, y - row_h * 0.3)
    return truncated


def build_supplementary_pdf(figA, figB, descriptions, table, title, out, tmp_prefix='./results/FigureS5'):
    """Panels A, B (with descriptions) + Table C -> two-page Letter PDF."""
    images = [export_png(figA, f'{tmp_prefix}_A.png'), export_png(figB, f'{tmp_prefix}_B.png')]
    c = canvas.Canvas(out, pagesize=letter)
    draw_panels(c, images, descriptions)
    c.showPage()
    truncated = draw_table(c, table, title)
    c.showPage()
    c.save()
    print(f'{out}: 2 pages, {len(table)} genes, {truncated} names truncated in the PDF (full names in the TSV)')

In [19]:
tableC_fmt = tableC.assign(Fitness=tableC.Fitness.map('{:.3f}'.format),
                          **{'log10(TPM+1)': tableC['log10(TPM+1)'].map('{:.3f}'.format)})
title = (f'Table. {len(tableC)} highly expressed genes (top {TOP_FRACTION:.0%}, log10(TPM+1) ≥ {cutoff:.2f}) '
         f'in the mean-effect category (fitness within ±0.5) under glucose')
n_counted = int(cog_counts.n_genes.sum())
descriptions = [
    f'<b>(A)</b> Fitness under glucose (set6IT057) versus expression (log10(TPM+1), mean of three control '
    f'replicates, GSE85475) for {len(fitness_tpm):,} genes. Colors indicate fitness categories: essential (≤ −2), '
    f'important (−2 to −0.5), mean-effect (−0.5 to 0.5) and fitness-enhancing (&gt; 0.5). Dashed horizontal line: '
    f'top {TOP_FRACTION:.0%} expression cutoff (log10(TPM+1) ≥ {cutoff:.2f}).',
    f'<b>(B)</b> COG functional categories of the {len(selected)} highly expressed genes in the mean-effect category. '
    f'Only genes with a single COG category are shown ({n_counted}); {n_no_cog} genes without COG assignment '
    f'and {len(multi_cogs)} with multiple categories ({"; ".join(multi_cogs)}) are listed in the table (page 2).',
]
build_supplementary_pdf(figA, figB, descriptions, tableC_fmt, title, './results/FigureS5.pdf')

./results/FigureS5.pdf: 2 pages, 168 genes, 19 names truncated in the PDF (full names in the TSV)


## Step 8. Expression filter: unfiltered vs highly expressed mean-effect genes

Without an expression filter, the mean-effect category covers most of the genome.
To test which functions are specific to highly expressed mean-effect genes, the top 5% mean-effect genes
(Step 3) are compared with the remaining mean-effect genes: Fisher's exact test per COG group and per
COG letter (genes with a single COG letter), Benjamini–Hochberg FDR within each level.
Output: `results/TableSX_COG_enrichment.tsv`.

In [20]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests


def annotate_cogs(df, genome):
    """Fitness/TPM table + genome (with COGs) -> single-letter COG genes with COG letter and group."""
    df = df.merge(genome[['locus_tag', 'func']], how='left', on='locus_tag')
    df = df[df.func.notna() & ~df.func.str.contains(',', na=False)]
    return df.merge(COG_TABLE[['COG', 'group']], how='left', left_on='func', right_on='COG')


def composition(df, col='group'):
    """Genes -> counts and percentage per category."""
    n = df[col].value_counts()
    return pd.DataFrame({'n_genes': n, 'percent': (100 * n / n.sum()).round(1)})


def cog_enrichment(df, col, flag):
    """Genes + category column + boolean flag -> Fisher test per category (flagged vs rest), BH FDR."""
    a_tot, b_tot = df[flag].sum(), (~df[flag]).sum()
    rows = []
    for k in sorted(df[col].dropna().unique()):
        a = ((df[col] == k) & df[flag]).sum()
        b = ((df[col] == k) & ~df[flag]).sum()
        odds, p = fisher_exact([[a, a_tot - a], [b, b_tot - b]])
        rows.append((k, a, 100 * a / a_tot, b, 100 * b / b_tot, odds, p))
    res = pd.DataFrame(rows, columns=['category', 'top_n', 'top_%', 'rest_n', 'rest_%', 'odds_ratio', 'p'])
    res['FDR'] = multipletests(res.p, method='fdr_bh')[1]
    return res.sort_values('p').reset_index(drop=True)

In [21]:
mean_effect = fitness_tpm[fitness_tpm.D_Glucose_cat == 'N'].assign(top=lambda d: d.TPM_log10 >= cutoff)
print(f'Mean-effect genes: {len(mean_effect)} of {len(fitness_tpm)} ({len(mean_effect) / len(fitness_tpm):.0%}); '
      f'top {TOP_FRACTION:.0%}: {mean_effect.top.sum()}')

me_cog = annotate_cogs(mean_effect, genome)
print(f'With a single COG letter: {len(me_cog)} (top: {me_cog.top.sum()}, rest: {(~me_cog.top).sum()})')
composition(me_cog).loc[GROUP_ORDER]

Mean-effect genes: 4210 of 4477 (94%); top 5%: 168
With a single COG letter: 3315 (top: 123, rest: 3192)


,n_genes,percent
INFORMATION,610,18.4
CELLULAR PROCESSES,894,27.0
METABOLISM,1435,43.3
POORLY CHARACTERIZED,376,11.3


In [22]:
enrich_group  = cog_enrichment(me_cog, 'group', 'top').assign(level='COG group')
enrich_letter = cog_enrichment(me_cog, 'func', 'top').assign(level='COG letter')
enrich_letter = enrich_letter.merge(COG_TABLE[['COG', 'name']], how='left', left_on='category', right_on='COG').drop(columns='COG')

enrichment = pd.concat([enrich_group, enrich_letter], ignore_index=True)
enrichment = enrichment[['level', 'category', 'name', 'top_n', 'top_%', 'rest_n', 'rest_%', 'odds_ratio', 'p', 'FDR']]
enrichment.to_csv('./results/TableSX_COG_enrichment.tsv', sep='\t', index=False, float_format='%.4g')
enrichment.round({'top_%': 1, 'rest_%': 1, 'odds_ratio': 2, 'p': 4, 'FDR': 4})

,level,category,name,top_n,top_%,rest_n,rest_%,odds_ratio,p,FDR
0,COG group,CELLULAR PROCESSES,NaN,44,35.8,850,26.6,1.53,0.0294,0.1177
1,COG group,METABOLISM,NaN,43,35.0,1392,43.6,0.70,0.0635,0.1270
2,COG group,POORLY CHARACTERIZED,NaN,11,8.9,365,11.4,0.76,0.4695,0.5544
3,COG group,INFORMATION,NaN,25,20.3,585,18.3,1.14,0.5544,0.5544
4,COG letter,O,Posttranslational,12,9.8,112,3.5,2.97,0.0018,0.0243
5,COG letter,M,Cell wall,16,13.0,176,5.5,2.56,0.0022,0.0243
6,COG letter,J,Translation,11,8.9,113,3.5,2.68,0.0056,0.0412
7,COG letter,H,Coenzyme,1,0.8,130,4.1,0.19,0.0921,0.5066
8,COG letter,R,General function,4,3.3,220,6.9,0.45,0.1412,0.5400
9,COG letter,D,Cell division,3,2.4,33,1.0,2.39,0.1473,0.5400
